# 19 · Embeddings and similarity search

**Prerequisites:** 00 — Setup; basic Python. No other notebook state is required.

**Execution:** portable. Run cells from top to bottom. Each notebook is independent.

## What you will learn

Compare lexical TF-IDF with latent SVD embeddings; normalize vectors; rank nearest neighbors.


## Step 1 — Build a local representation

LSA uses truncated SVD over TF-IDF to learn a small latent representation. It is an offline teaching baseline, not a pretrained semantic embedding model.


In [ ]:
documents = [
    {'id':'delta', 'text':'Delta Lake supports ACID transactions and table version history.'},
    {'id':'catalog', 'text':'Unity Catalog controls permissions and governs tables and models.'},
    {'id':'mlflow', 'text':'MLflow tracks experiments, parameters, metrics, and model artifacts.'},
    {'id':'spark', 'text':'Spark processes large datasets using distributed DataFrame operations.'},
    {'id':'rag', 'text':'Retrieval augmented generation retrieves evidence before generating an answer.'},
    {'id':'batch', 'text':'Batch inference scores many records together on a schedule.'},
]

import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
vectorizer = TfidfVectorizer(ngram_range=(1, 2))
matrix = vectorizer.fit_transform([d['text'] for d in documents])
def retrieve(question, k=2):
    scores = cosine_similarity(vectorizer.transform([question]), matrix)[0]
    return [dict(documents[i], score=float(scores[i]))
            for i in np.argsort(-scores)[:k] if scores[i] > 0]

from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize
svd=TruncatedSVD(n_components=5,random_state=42)
embeddings=normalize(svd.fit_transform(matrix))
print('Dense embedding matrix:',embeddings.shape)


**Check your result:** Six documents have five-dimensional vectors. With very few components, unrelated documents in this tiny corpus can collapse to the same direction; try four components in the exercise.


## Step 2 — Retrieve nearest vectors

Queries and documents must use the same fitted transformation. Dot products equal cosine similarity for normalized vectors.


In [ ]:
query='What tracks model experiments?'
q=normalize(svd.transform(vectorizer.transform([query])))
scores=(q@embeddings.T)[0]
rank=np.argsort(-scores)[:3]
print([(documents[i]['id'],round(float(scores[i]),3)) for i in rank])
assert documents[rank[0]]['id']=='mlflow'


**Check your result:** The MLflow document should rank first.


## Step 3 — Optionally call a real embedding endpoint

Choose a workspace embedding endpoint that you can query. The request contains only course synthetic text. Install databricks-sdk and openai if needed.


In [ ]:
RUN_EMBEDDING_API=False
EMBEDDING_ENDPOINT=''
if RUN_EMBEDDING_API:
    assert EMBEDDING_ENDPOINT
    from databricks.sdk import WorkspaceClient
    client=WorkspaceClient().serving_endpoints.get_open_ai_client()
    response=client.embeddings.create(model=EMBEDDING_ENDPOINT,input=[d['text'] for d in documents]+[query])
    vectors=np.array([v.embedding for v in sorted(response.data,key=lambda v:v.index)])
    vectors=normalize(vectors)
    print('API dimension:',vectors.shape[1],'scores:',vectors[-1]@vectors[:-1].T)
else:
    print('Live embeddings skipped; LSA lab completed.')


**Check your result:** Default mode performs no endpoint calls.


## Practice

Compare keyword retrieval and real embeddings on paraphrases with little word overlap.

Predict the outcome before changing the code. Explain the result in your own words.


## Solution guide

Build labeled query/document pairs and evaluate Recall@k. Rebuild the document index whenever the embedding model or preprocessing changes.

## Before using this at work

Use representative data, measure quality on held-out examples, record dependencies and permissions, and review failures. Tiny synthetic results demonstrate mechanics; they do not establish production quality.
